# 🚀 Natural-Language to SQL using MySQL HeatWave
### 🔥 Demo for querying MySQL HeatWave  database using Natural Language
Visit : https://blogs.oracle.com/mysql/introducing-natural-language-to-sql-for-mysql-heatwave
for more info

### Setup
#### Install required dependencies

In [54]:
!pip install pandas sqlglot mysql-connector-python matplotlib ipywidgets tabulate

10564.09s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Looking in indexes: https://artifactory.oci.oraclecorp.com/api/pypi/global-release-pypi/simple


#### Setup environment variables to connect to MySQL instance

In [ ]:
from IPython.display import Markdown, display
import json
import pandas as pd
import sqlglot
from pprint import pprint
import mysql.connector
import os

# Replace
os.environ['USER'] = 'root'
os.environ['USER_PASSWORD'] = ''
os.environ['HOST_IP'] = '127.0.0.1'

#### Setup some helper methods

In [ ]:
def fetch_output(cursor):
    for _, result_set in cursor.fetchsets():
        if len(result_set)==0 or result_set is None:
            continue
        r = result_set[0][0]
        try:
            r = json.loads(r)
            if "level" in r: # log printing
                if "Generated" in r['message']:
                    log_message, sql_query = tuple(r['message'].split(':'))
                    sql_query = sqlglot.parse_one(sql_query, read='mysql').sql(pretty=True,identify=True, dialect='mysql')
                    display(Markdown(f"*{log_message}*"))
                    display(Markdown(f"```sql\n{sql_query}"))
                else:
                    display(Markdown(f"*{r['message']}*")) 
        except Exception: # result set parsing
            df = pd.DataFrame(result_set, columns=cursor.column_names)
            o_res = df.to_markdown()
            display(Markdown(f"{o_res}"))
    return df

def pretty_print_output(out_args):
    # Formatting the JSON object returned by the stored procedure
    output = json.loads(out_args[1] if isinstance(out_args, tuple) else out_args)
    pprint(output)

#### Load airport-db
See instructions here : https://dev.mysql.com/doc/heatwave/en/mys-hw-airportdb-quickstart.html

In [3]:
# Setup connection
mydb_connection = mysql.connector.connect(
            user=os.environ['USER'],
            password=os.environ['USER_PASSWORD'],
            host=os.environ['HOST_IP'],
            port=3306,
            autocommit=True,
            database="airportdb"
        )
cursor = mydb_connection.cursor(buffered=True)

In [4]:
cursor.execute("SHOW TABLES")
cursor.fetchall()

[('airline',),
 ('airplane',),
 ('airplane_type',),
 ('airport',),
 ('airport_geo',),
 ('airport_reachable',),
 ('booking',),
 ('employee',),
 ('flight',),
 ('flight_log',),
 ('flightschedule',),
 ('passenger',),
 ('passengerdetails',),
 ('weatherdata',)]

In [ ]:
nlq = "Show me the total number of tickets with price more than $200"
cursor.execute("CALL sys.NL_SQL(%s, @out, NULL);",[nlq])

In [9]:
_  = fetch_output(cursor)

*Generated SQL statement*

```sql
SELECT
  COUNT(`booking_id`)
FROM `airportdb`.`booking`
WHERE
  `price` > 200

*Executing generated SQL statement...*

|    |   COUNT(`booking_id`) |
|---:|----------------------:|
|  0 |           3.26991e+07 |

In [10]:
nlq = "Show me the total number of tickets with price more than $200, and the average price"
cursor.execute("CALL sys.NL_SQL(%s, @out, NULL)", [nlq])
_ = fetch_output(cursor)

*Generated SQL statement*

```sql
SELECT
  COUNT(`booking_id`),
  AVG(`price`)
FROM `airportdb`.`booking`
WHERE
  `price` > 200

*Executing generated SQL statement...*

|    |   COUNT(`booking_id`) |   AVG(`price`) |
|---:|----------------------:|---------------:|
|  0 |              32699080 |        350.405 |

In [11]:
nlq = "List five airlines that have the highest number of aircrafts along with the total aircrafts they have"
cursor.execute("CALL sys.NL_SQL(%s, @out, NULL)", [nlq])

In [12]:
df = fetch_output(cursor)

*Generated SQL statement*

```sql
SELECT
  `T1`.`airlinename`,
  COUNT(`T2`.`airplane_id`) AS `total_aircrafts`
FROM `airportdb`.`airline` AS `T1`
JOIN `airportdb`.`airplane` AS `T2`
  ON `T1`.`airline_id` = `T2`.`airline_id`
GROUP BY
  `T1`.`airlinename`
ORDER BY
  `total_aircrafts` DESC
LIMIT 5

*Executing generated SQL statement...*

|    | airlinename       |   total_aircrafts |
|---:|:------------------|------------------:|
|  0 | Oman Airlines     |                90 |
|  1 | Djibouti Airlines |                89 |
|  2 | Syria Airlines    |                89 |
|  3 | Iceland Airlines  |                89 |
|  4 | Romania Airlines  |                89 |

In [13]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
# Dropdown widgets for axis selection
x_dropdown = widgets.Dropdown(
    options=df.columns,
    value=df.columns[0],
    description='X Axis:'
)
y_dropdown = widgets.Dropdown(
    options=df.columns,
    value=df.columns[1],
    description='Y Axis:'
)

def plot_bar_chart(x_col, y_col):
    plt.figure(figsize=(8, 5))
    plt.bar(df[x_col], df[y_col])
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f'Bar Chart of {y_col} vs {x_col}')
    plt.show()

# Display interactive widget
widgets.interact(plot_bar_chart, x_col=x_dropdown, y_col=y_dropdown)

interactive(children=(Dropdown(description='X Axis:', options=('airlinename', 'total_aircrafts'), value='airli…

<function __main__.plot_bar_chart(x_col, y_col)>

In [14]:
cursor.execute(f"SELECT @out")
res =  cursor.fetchall()

In [15]:
pretty_print_output(res[0][0])

{'is_sql_valid': 1,
 'model_id': 'meta.llama-4-maverick-17b-128e-instruct-fp8',
 'schemas': ['airportdb'],
 'sql_query': 'SELECT `T1`.`airlinename`, COUNT(`T2`.`airplane_id`) AS '
              '`total_aircrafts` FROM `airportdb`.`airline` AS `T1` JOIN '
              '`airportdb`.`airplane` AS `T2` ON `T1`.`airline_id` = '
              '`T2`.`airline_id` GROUP BY `T1`.`airlinename` ORDER BY '
              '`total_aircrafts` DESC LIMIT 5',
 'tables': ['airportdb.airplane',
            'airportdb.airplane_type',
            'airportdb.flight',
            'airportdb.airline',
            'airportdb.airport_reachable',
            'airportdb.flightschedule',
            'airportdb.flight_log',
            'airportdb.airport_geo',
            'airportdb.booking',
            'airportdb.airport',
            'airportdb.passenger',
            'airportdb.passengerdetails']}
